Cell 1 — Imports & Setup

In [1]:
import os
import sqlite3
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

# Paths
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)

Cell 2 — Load Data

In [2]:
print("Files in raw folder:", os.listdir(RAW_DIR))

movies = pd.read_csv(RAW_DIR / 'movies.csv')
ratings = pd.read_csv(RAW_DIR / 'ratings.csv')

print("\n--- Movies Data ---")
display(movies.head())
print("\n--- Ratings Data ---")
display(ratings.head())

# Backups (used later for before/after comparison)
movies_raw = movies.copy()
ratings_raw = ratings.copy()

Files in raw folder: ['movies.csv', 'ratings.csv']

--- Movies Data ---


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



--- Ratings Data ---


,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


Cell 3 — Diagnostic Report

In [3]:
print("\n" + "=" * 60)
print("DIAGNOSTIC REPORT")
print("=" * 60)

print(f"Movies shape: {movies.shape}")
print(f"Ratings shape: {ratings.shape}")

print(f"\nMovies - NaN:\n{movies.isnull().sum()}")
print(f"\nRatings - NaN:\n{ratings.isnull().sum()}")

print(f"\nMovies - duplicated movieId: {movies['movieId'].duplicated().sum()}")
print(f"Movies - duplicated title  : {movies['title'].duplicated().sum()}")
print(f"Ratings - duplicated (user, movie): {ratings.duplicated(subset=['userId', 'movieId']).sum()}")
print(f"\nRatings - rating range: [{ratings['rating'].min()}, {ratings['rating'].max()}]")


DIAGNOSTIC REPORT
Movies shape: (10329, 3)
Ratings shape: (105339, 4)

Movies - NaN:
movieId    0
title      0
genres     0
dtype: int64

Ratings - NaN:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Movies - duplicated movieId: 0
Movies - duplicated title  : 2
Ratings - duplicated (user, movie): 0

Ratings - rating range: [0.5, 5.0]


Cell 4 — Data Types & Statistical Summary

In [4]:
print("\n" + "-" * 70)
print("Movies Data Types:")
print("-" * 70)
print(movies.dtypes)

print("\n" + "-" * 70)
print("Ratings Data Types:")
print("-" * 70)
print(ratings.dtypes)

print("\n" + "=" * 70)
print("Movies Info (Types + Memory):")
print("=" * 70)
movies.info()

print("\n" + "=" * 70)
print("Ratings Info (Types + Memory):")
print("=" * 70)
ratings.info()

print("\n" + "=" * 70)
print("STATISTICAL SUMMARY")
print("=" * 70)

print("\nMovies - Describe (include='all'):")
display(movies.describe(include='all'))

print("\nRatings - Describe:")
display(ratings.describe())

print("\nRatings - Describe with custom percentiles [1%, 5%, 25%, 50%, 75%, 95%, 99%]:")
display(ratings.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\n" + "=" * 70)
print("UNIQUE VALUES COUNT")
print("=" * 70)

print("\nMovies - Nunique:")
print(movies.nunique())

print("\nRatings - Nunique:")
print(ratings.nunique())


----------------------------------------------------------------------
Movies Data Types:
----------------------------------------------------------------------
movieId     int64
title      object
genres     object
dtype: object

----------------------------------------------------------------------
Ratings Data Types:
----------------------------------------------------------------------
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

Movies Info (Types + Memory):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB

Ratings Info (Types + Memory):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 4 c

,movieId,title,genres
count,10329.000000,10329,10329
unique,NaN,10327,938
top,NaN,Men with Guns (1997),Drama
freq,NaN,2,1385
mean,31924.282893,NaN,NaN
std,37734.741149,NaN,NaN
min,1.000000,NaN,NaN
25%,3240.000000,NaN,NaN
50%,7088.000000,NaN,NaN
75%,59900.000000,NaN,NaN



Ratings - Describe:


,userId,movieId,rating,timestamp
count,105339.000000,105339.000000,105339.000000,1.053390e+05
mean,364.924539,13381.312477,3.516850,1.130424e+09
std,197.486905,26170.456869,1.044872,1.802660e+08
min,1.000000,1.000000,0.500000,8.285650e+08
25%,192.000000,1073.000000,3.000000,9.711008e+08
50%,383.000000,2497.000000,3.500000,1.115154e+09
75%,557.000000,5991.000000,4.000000,1.275496e+09
max,668.000000,149532.000000,5.000000,1.452405e+09



Ratings - Describe with custom percentiles [1%, 5%, 25%, 50%, 75%, 95%, 99%]:


,userId,movieId,rating,timestamp
count,105339.000000,105339.000000,105339.000000,1.053390e+05
mean,364.924539,13381.312477,3.516850,1.130424e+09
std,197.486905,26170.456869,1.044872,1.802660e+08
min,1.000000,1.000000,0.500000,8.285650e+08
1%,15.000000,17.000000,0.500000,8.345976e+08
5%,60.000000,161.000000,1.500000,8.484051e+08
25%,192.000000,1073.000000,3.000000,9.711008e+08
50%,383.000000,2497.000000,3.500000,1.115154e+09
75%,557.000000,5991.000000,4.000000,1.275496e+09
95%,668.000000,79242.000000,5.000000,1.440812e+09



UNIQUE VALUES COUNT

Movies - Nunique:
movieId    10329
title      10327
genres       938
dtype: int64

Ratings - Nunique:
userId         668
movieId      10325
rating          10
timestamp    84686
dtype: int64


Cell 5 — Cleaning (Missing Values, Timestamp, Duplicates, Range)

In [5]:
print("\n" + "=" * 60)
print("CLEANING")
print("=" * 60)

# 1. Missing values
n_before = len(ratings)
ratings = ratings.dropna(subset=['userId', 'movieId', 'rating']).copy()
print(f"Ratings: removed {n_before - len(ratings)} rows due to NaN")

n_before = len(movies)
movies = movies.dropna(subset=['movieId', 'title']).copy()
print(f"Movies : removed {n_before - len(movies)} rows due to NaN")

movies['genres'] = movies['genres'].fillna('Unknown')
movies['genres'] = movies['genres'].str.replace('(no genres listed)', 'Unknown', regex=False)
print("Genres : replaced missing and '(no genres listed)' with 'Unknown'")

# 2. Timestamp
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year'] = ratings['timestamp'].dt.year
ratings['month'] = ratings['timestamp'].dt.month
print("Timestamp: converted to datetime and extracted year & month")

# 3. Duplicates
# 3a. Movies: true duplicates are repeated movieIds (the primary key)
n_before = len(movies)
movies = movies.drop_duplicates(subset='movieId', keep='first').reset_index(drop=True)
print(f"Movies: removed {n_before - len(movies)} duplicate movieId rows")

# 3b. Same title under a different movieId is NOT an error (remakes / different
#     films with the same name). Dropping one would silently delete its ratings,
#     so we keep both and make the titles unique instead.
dup_title = movies['title'].duplicated(keep=False)
if dup_title.any():
    movies.loc[dup_title, 'title'] = (
        movies.loc[dup_title, 'title'] + ' [movieId ' + movies.loc[dup_title, 'movieId'].astype(str) + ']'
    )
    print(f"Movies: {dup_title.sum()} rows shared a title with another movieId -> kept, title suffixed with movieId")
else:
    print("Movies: no duplicate titles found")

# 3c. Ratings that point to a movie missing from movies.csv
valid_ids = set(movies['movieId'])
n_before = len(ratings)
ratings = ratings[ratings['movieId'].isin(valid_ids)].copy()
print(f"Ratings: removed {n_before - len(ratings)} ratings for movies not in movies.csv")

# 3d. Duplicate (user, movie) pairs: keep the MOST RECENT rating
#     (sort by timestamp first, otherwise keep='last' just means "last row in the file")
n_before = len(ratings)
ratings = (
    ratings.sort_values('timestamp')
    .drop_duplicates(subset=['userId', 'movieId'], keep='last')
    .sort_values(['userId', 'movieId'])
    .reset_index(drop=True)
)
print(f"Ratings: removed {n_before - len(ratings)} duplicate (user, movie) pairs (kept latest)")

# 4. Rating range
out_of_range = ratings[(ratings['rating'] < 0.5) | (ratings['rating'] > 5.0)]
if len(out_of_range) > 0:
    print(f"Ratings: removing {len(out_of_range)} out-of-range ratings")
    ratings = ratings[(ratings['rating'] >= 0.5) & (ratings['rating'] <= 5.0)].copy()
else:
    print("Ratings: all ratings within [0.5, 5.0]")

print("\n" + "=" * 60)
print("CLEANING COMPLETED")
print("=" * 60)
print(f"Final Ratings : {len(ratings)} rows")
print(f"Final Movies  : {len(movies)} rows")


CLEANING
Ratings: removed 0 rows due to NaN
Movies : removed 0 rows due to NaN
Genres : replaced missing and '(no genres listed)' with 'Unknown'
Timestamp: converted to datetime and extracted year & month
Movies: removed 0 duplicate movieId rows
Movies: 4 rows shared a title with another movieId -> kept, title suffixed with movieId
Ratings: removed 0 ratings for movies not in movies.csv
Ratings: removed 0 duplicate (user, movie) pairs (kept latest)
Ratings: all ratings within [0.5, 5.0]

CLEANING COMPLETED
Final Ratings : 105339 rows
Final Movies  : 10329 rows


Cell 6 — Filtering (Min Ratings per User / Movie)

In [6]:
print("\n" + "=" * 60)
print("FILTERING")
print("=" * 60)

MIN_USER_RATINGS = 2
MIN_MOVIE_RATINGS = 2

before_users = ratings['userId'].nunique()
before_movies = ratings['movieId'].nunique()

# Filter until stable: removing movies can push a user below the threshold
# (and vice-versa), so a single pass does not guarantee the thresholds hold.
n_passes = 0
while True:
    n_passes += 1
    n_before = len(ratings)

    user_counts = ratings['userId'].value_counts()
    ratings = ratings[ratings['userId'].map(user_counts) >= MIN_USER_RATINGS]

    movie_counts = ratings['movieId'].value_counts()
    ratings = ratings[ratings['movieId'].map(movie_counts) >= MIN_MOVIE_RATINGS]

    if len(ratings) == n_before:
        break

ratings = ratings.reset_index(drop=True)
movies = movies[movies['movieId'].isin(set(ratings['movieId']))].reset_index(drop=True)

print(f"Filtering converged after {n_passes} pass(es)")
print(f"Users : {before_users} -> {ratings['userId'].nunique()}")
print(f"Movies: {before_movies} -> {ratings['movieId'].nunique()}")
print(f"Final Ratings: {len(ratings)} rows")
print(f"Final Movies : {len(movies)} rows")


FILTERING
Filtering converged after 2 pass(es)
Users : 668 -> 668
Movies: 10325 -> 6687
Final Ratings: 101701 rows
Final Movies : 6687 rows


Cell 7 — Outlier Inspection (IQR-based)

In [7]:
print("\n" + "=" * 60)
print("OUTLIER INSPECTION (IQR Method)")
print("=" * 60)

user_counts = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()

q1_u, q3_u = user_counts.quantile([0.25, 0.75])
q1_m, q3_m = movie_counts.quantile([0.25, 0.75])
iqr_u, iqr_m = q3_u - q1_u, q3_m - q1_m

power_users = user_counts[user_counts > q3_u + 1.5 * iqr_u]
blockbusters = movie_counts[movie_counts > q3_m + 1.5 * iqr_m]

flat_mask = ratings.groupby('userId')['rating'].transform('std').fillna(0) == 0
flat_raters = ratings[flat_mask]

print(f"Power users (high activity)   : {power_users.sum()} ratings "
      f"({power_users.sum() / len(ratings) * 100:.1f}%)")
print(f"Blockbusters (high popularity): {blockbusters.sum()} ratings "
      f"({blockbusters.sum() / len(ratings) * 100:.1f}%)")
print(f"Flat raters (constant rating) : {flat_raters.shape[0]} ratings "
      f"({flat_raters.shape[0] / len(ratings) * 100:.1f}%)")

print("\nOutliers were only inspected here: nothing is removed and no flag column is saved.")
print("(For recommenders, heavy users / popular movies are real behaviour, not errors.)")


OUTLIER INSPECTION (IQR Method)
Power users (high activity)   : 49202 ratings (48.4%)
Blockbusters (high popularity): 59364 ratings (58.4%)
Flat raters (constant rating) : 35 ratings (0.0%)

Outliers were only inspected here: nothing is removed and no flag column is saved.
(For recommenders, heavy users / popular movies are real behaviour, not errors.)


Cell 8 — Structured Storage (CSV + SQLite)

In [8]:
print("\n" + "=" * 60)
print("STRUCTURED STORAGE")
print("=" * 60)

# 1. CSV
ratings.to_csv(PROCESSED_DIR / 'ratings_clean.csv', index=False)
movies.to_csv(PROCESSED_DIR / 'movies_clean.csv', index=False)
print("Saved ratings_clean.csv and movies_clean.csv")

# 2. SQLite (always close the connection, even if a write fails)
conn = sqlite3.connect(PROCESSED_DIR / 'movielens.db')
try:
    ratings.to_sql('ratings_clean', conn, if_exists='replace', index=False)
    movies.to_sql('movies_clean', conn, if_exists='replace', index=False)
finally:
    conn.close()
print("Saved to SQLite (movielens.db)")


STRUCTURED STORAGE
Saved ratings_clean.csv and movies_clean.csv
Saved to SQLite (movielens.db)


In [9]:
# Cell 8.5 — Save reusable genre exploded dataset
movies_exploded = movies.copy()
movies_exploded['genres'] = movies_exploded['genres'].str.split('|')
movies_exploded = movies_exploded.explode('genres').reset_index(drop=True)

# Merge with movie stats (avg rating + count)
movie_stats = (
    ratings.groupby('movieId')['rating']
    .agg(avg_rating='mean', num_ratings='count')
    .reset_index()
)
genre_movies = movies_exploded.merge(movie_stats, on='movieId', how='left')

# Save
genre_movies.to_csv(PROCESSED_DIR / 'genre_movies.csv', index=False)
conn = sqlite3.connect(PROCESSED_DIR / 'movielens.db')
try:
    genre_movies.to_sql('genre_movies', conn, if_exists='replace', index=False)
finally:
    conn.close()
print(f"Saved genre_movies.csv: {genre_movies.shape}")

Saved genre_movies.csv: (15747, 5)


Cell 9 — PySpark Demonstration

In [10]:
print("\n" + "=" * 60)
print("PYSPARK DEMONSTRATION")
print("=" * 60)

spark = None
try:
    from pyspark.sql import SparkSession
    import pyspark.sql.functions as F

    spark = (
        SparkSession.builder
        .appName("MovieLens_Cleaning")
        .master("local[*]")
        .getOrCreate()
    )
    print(f"Spark version: {spark.version}")

    movies_spark = spark.read.csv(str(RAW_DIR / 'movies.csv'), header=True, inferSchema=True)
    ratings_spark = spark.read.csv(str(RAW_DIR / 'ratings.csv'), header=True, inferSchema=True)
    print(f"Movies (Spark): {movies_spark.count()} rows")
    print(f"Ratings (Spark): {ratings_spark.count()} rows")

    print("\nTop 5 Movies (Avg Rating with >= 50 ratings):")
    (
        ratings_spark.groupBy("movieId")
        .agg(F.mean("rating").alias("avg_rating"), F.count("rating").alias("num_ratings"))
        .filter(F.col("num_ratings") >= 50)
        .orderBy(F.desc("avg_rating"))
        .limit(5)
        .show()
    )
    print("PySpark demonstration completed successfully.")

except ImportError:
    print("PySpark is not installed (pip install pyspark).")
    print("Skipping this section; the code is ready to run.")
except Exception as e:
    # e.g. Java missing / JAVA_HOME not set -> don't break the whole notebook
    print(f"Spark could not run in this environment: {type(e).__name__}: {e}")
    print("Skipping this section.")
finally:
    if spark is not None:
        spark.stop()


PYSPARK DEMONSTRATION
Spark version: 4.2.0
Movies (Spark): 10329 rows
Ratings (Spark): 105339 rows

Top 5 Movies (Avg Rating with >= 50 ratings):
+-------+-----------------+-----------+
|movieId|       avg_rating|num_ratings|
+-------+-----------------+-----------+
|    318|4.454545454545454|        308|
|    923|4.396103896103896|         77|
|    858|4.392857142857143|        210|
|   3000|4.384615384615385|         52|
|    904|4.331081081081081|         74|
+-------+-----------------+-----------+

PySpark demonstration completed successfully.


Cell 10 — Before vs After Summary

In [11]:
print("\n" + "=" * 60)
print("BEFORE vs AFTER")
print("=" * 60)

summary = pd.DataFrame({
    'Metric': ['Ratings', 'Users', 'Movies (in ratings)', 'Movies rows'],
    'Before': [
        len(ratings_raw),
        ratings_raw['userId'].nunique(),
        ratings_raw['movieId'].nunique(),
        len(movies_raw),
    ],
    'After': [
        len(ratings),
        ratings['userId'].nunique(),
        ratings['movieId'].nunique(),
        len(movies),
    ],
})
summary['Lost %'] = (
    (summary['Before'] - summary['After']) / summary['Before'] * 100
).round(2)
display(summary)


BEFORE vs AFTER


,Metric,Before,After,Lost %
0,Ratings,105339,101701,3.45
1,Users,668,668,0.00
2,Movies (in ratings),10325,6687,35.23
3,Movies rows,10329,6687,35.26
